# 🧩 Mini-Lab: Testability Map

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** the typical stages of an AI pipeline and what each stage does
2. **Classify** each stage as deterministic or stochastic
3. **Decide** which parts of an AI application deserve traditional unit tests vs. other evaluation strategies

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Deterministic vs Stochastic Behavior | Test what your code does with outputs you control, and measure model quality separately. Treat model sampling as stochastic, isolate it behind a boundary so unit tests stay reliable. |
| AI Pipeline Stages | A practical AI feature pipeline with six boxes: ingest, prepare, call, validate, log, evaluate. The model call is just one box, not the whole system. |


## The Problem: What Should You Actually Test?

An AI application is **not** one big blob. It is a pipeline of stages:

```
User Input → Validate → Build Prompt → Call LLM → Parse Output → Format Response
```

Some of these stages are **deterministic** — same input, same output, every time. Others are **stochastic** — even with the same input, the output can change.

The key insight: **you can write reliable, traditional tests for the deterministic stages** (and there are more of them than you think).

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## A Concrete AI Pipeline

Let's look at a simple document-extraction pipeline. The user sends raw text and gets back structured JSON. We'll define each stage as a small function.

In [2]:
import json

# ── Stage 1: Validate Input ──────────────────────────────────
def validate_input(text: str) -> str:
    """Reject empty or too-long inputs."""
    text = text.strip()
    if not text:
        raise ValueError("Input text is empty.")
    if len(text) > 5000:
        raise ValueError("Input text exceeds 5000 characters.")
    return text


# ── Stage 2: Build Prompt ────────────────────────────────────
def build_prompt(text: str) -> list[dict]:
    """Create the messages list for the LLM call."""
    return [
        {"role": "system", "content": "Extract name, email, and company from the text. Reply in JSON."},
        {"role": "user", "content": text},
    ]


# ── Stage 3: Call LLM (simulated) ────────────────────────────
def call_llm(messages: list[dict]) -> str:
    """Simulate an LLM response. In production this hits an API."""
    # Simulated — a real call would be non-deterministic
    return '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}'


# ── Stage 4: Parse Output ────────────────────────────────────
def parse_output(raw: str) -> dict:
    """Parse JSON from the LLM's raw string response."""
    return json.loads(raw)


# ── Stage 5: Format Response ─────────────────────────────────
def format_response(data: dict) -> str:
    """Turn parsed data into a user-friendly string."""
    parts = [f"- **{k.title()}**: {v}" for k, v in data.items()]
    return "\n".join(parts)


print("Pipeline functions defined.")

Pipeline functions defined.


## Mapping Each Stage

Now let's classify every stage. The question for each one is simple:

> **Given the exact same input, will this function *always* return the exact same output?**

If yes → **deterministic** → unit-testable.  
If no → **stochastic** → needs a different evaluation strategy.

In [3]:
pipeline_stages = [
    {
        "stage": "1. Validate Input",
        "function": "validate_input()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "Pure string logic — same input always gives same result or same error.",
    },
    {
        "stage": "2. Build Prompt",
        "function": "build_prompt()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "String formatting — template + input always produces the same messages list.",
    },
    {
        "stage": "3. Call LLM",
        "function": "call_llm()",
        "behavior": "Stochastic",
        "testable_with_pytest": "⚠️ Not directly",
        "why": "Model output varies across runs. Needs evaluation metrics, not assert ==.",
    },
    {
        "stage": "4. Parse Output",
        "function": "parse_output()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "JSON parsing is pure code — same string always yields the same dict (or same error).",
    },
    {
        "stage": "5. Format Response",
        "function": "format_response()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "String formatting — same dict always produces the same output string.",
    },
]

# Build a markdown table
header = "| Stage | Function | Behavior | Testable with pytest? | Why |\n"
header += "|-------|----------|----------|-----------------------|-----|\n"
rows = ""
for s in pipeline_stages:
    rows += f"| {s['stage']} | `{s['function']}` | **{s['behavior']}** | {s['testable_with_pytest']} | {s['why']} |\n"

md(header + rows)

| Stage | Function | Behavior | Testable with pytest? | Why |
|-------|----------|----------|-----------------------|-----|
| 1. Validate Input | `validate_input()` | **Deterministic** | ✅ Yes | Pure string logic — same input always gives same result or same error. |
| 2. Build Prompt | `build_prompt()` | **Deterministic** | ✅ Yes | String formatting — template + input always produces the same messages list. |
| 3. Call LLM | `call_llm()` | **Stochastic** | ⚠️ Not directly | Model output varies across runs. Needs evaluation metrics, not assert ==. |
| 4. Parse Output | `parse_output()` | **Deterministic** | ✅ Yes | JSON parsing is pure code — same string always yields the same dict (or same error). |
| 5. Format Response | `format_response()` | **Deterministic** | ✅ Yes | String formatting — same dict always produces the same output string. |


## Proving It: Deterministic Stages Are Predictable

Let's run the deterministic stages multiple times and confirm the output never changes.

In [4]:
sample_text = "  Contact Alice Smith at alice@example.com, she works at Acme Inc.  "

# Run each deterministic stage 5 times and check consistency
deterministic_stages = {
    "validate_input": lambda: validate_input(sample_text),
    "build_prompt": lambda: build_prompt(sample_text.strip()),
    "parse_output": lambda: parse_output('{"name": "Alice"}'),
    "format_response": lambda: format_response({"name": "Alice", "email": "alice@example.com"}),
}

for name, fn in deterministic_stages.items():
    results = [fn() for _ in range(5)]
    all_same = all(r == results[0] for r in results)
    print(f"{name:20s} → 5 runs identical? {all_same}")

validate_input       → 5 runs identical? True
build_prompt         → 5 runs identical? True
parse_output         → 5 runs identical? True
format_response      → 5 runs identical? True


Every deterministic stage returns the exact same output across all runs. These are the stages you can pin down with simple `assert` statements in pytest.

## The Stochastic Stage: Why `assert ==` Doesn't Work

The LLM call is different. Even with `temperature=0`, outputs can vary slightly across API versions or model updates. Let's simulate this.

In [5]:
import random

def call_llm_realistic(messages: list[dict]) -> str:
    """Simulate slight non-determinism in LLM responses."""
    variations = [
        '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}',
        '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}',
        '{"name":"Alice Smith","email":"alice@example.com","company":"Acme"}',
    ]
    return random.choice(variations)

messages = build_prompt("Contact Alice Smith at alice@example.com, Acme Inc.")
responses = [call_llm_realistic(messages) for _ in range(6)]

print("Six simulated LLM responses:")
for i, r in enumerate(responses, 1):
    print(f"  Run {i}: {r}")

print(f"\nAll identical? {len(set(responses)) == 1}")

Six simulated LLM responses:
  Run 1: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 2: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}
  Run 3: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}
  Run 4: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 5: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 6: {"name":"Alice Smith","email":"alice@example.com","company":"Acme"}

All identical? False


The LLM stage may return slightly different strings each time. You **cannot** write `assert response == expected` and expect it to stay green. That stage requires evaluation strategies (e.g., checking required keys exist, semantic similarity checks) — not exact-match unit tests.

## The Testability Map

Here's the mental model to carry forward:

In [6]:
testability_map = """
## 🗺️ Testability Map

```
┌──────────────────────────────────────────────────────────────┐
│                    AI Pipeline Stages                       │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ✅ DETERMINISTIC (unit-testable)                            │
│  ┌────────────┐  ┌─────────────┐  ┌──────────────┐          │
│  │  Validate   │  │ Build Prompt │  │ Parse Output │          │
│  │   Input     │  │  (template)  │  │   (JSON)     │          │
│  └────────────┘  └─────────────┘  └──────────────┘          │
│  ┌─────────────────┐                                         │
│  │ Format Response  │                                         │
│  └─────────────────┘                                         │
│                                                              │
│  ⚠️  STOCHASTIC (needs eval, not assert ==)                  │
│  ┌────────────┐                                              │
│  │  Call LLM   │  ← output varies run to run                │
│  └────────────┘                                              │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

**Rule of thumb:** Most of your AI app is deterministic glue code.
Test that glue thoroughly, and evaluate the LLM part separately.
"""

md(testability_map)


## 🗺️ Testability Map

```
┌──────────────────────────────────────────────────────────────┐
│                    AI Pipeline Stages                       │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ✅ DETERMINISTIC (unit-testable)                            │
│  ┌────────────┐  ┌─────────────┐  ┌──────────────┐          │
│  │  Validate   │  │ Build Prompt │  │ Parse Output │          │
│  │   Input     │  │  (template)  │  │   (JSON)     │          │
│  └────────────┘  └─────────────┘  └──────────────┘          │
│  ┌─────────────────┐                                         │
│  │ Format Response  │                                         │
│  └─────────────────┘                                         │
│                                                              │
│  ⚠️  STOCHASTIC (needs eval, not assert ==)                  │
│  ┌────────────┐                                              │
│  │  Call LLM   │  ← output varies run to run                │
│  └────────────┘                                              │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

**Rule of thumb:** Most of your AI app is deterministic glue code.
Test that glue thoroughly, and evaluate the LLM part separately.


## 🎯 Summary

### Key Takeaways

1. **AI pipelines have distinct stages** — input validation, prompt building, LLM calling, output parsing, and response formatting are separate, testable units
2. **Most stages are deterministic** — same input produces the same output every time, making them perfect candidates for traditional `assert`-based unit tests
3. **Only the LLM call is stochastic** — model output can vary across runs, so exact-match assertions will produce flaky tests
4. **Test the glue, evaluate the model** — write pytest tests for all the deterministic code around the LLM, and use evaluation strategies (schema checks, semantic similarity) for the model output itself